<a href="https://colab.research.google.com/github/AndreaValenciaBorja/icu-billing-denial-prediction/blob/main/GridSearch/SMOTE/Xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **XGBoost**

### **Librerías**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_excel('df_nuevo.xlsx')

In [ ]:
X = data.drop('Glosa', axis=1)
y = data['Glosa']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print(f"\n📋 DIVISIÓN DE DATOS:")
print(f"• Entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(data)*100:.1f}%)")
print(f"• Prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/len(data)*100:.1f}%)")
print(f"\n🎯 Distribución en entrenamiento:")
print(y_train.value_counts())
print(f"🎯 Distribución en prueba:")
print(y_test.value_counts())


📋 DIVISIÓN DE DATOS:
• Entrenamiento: 2144 muestras (70.0%)
• Prueba: 920 muestras (30.0%)

🎯 Distribución en entrenamiento:
Glosa
0    1800
1     344
Name: count, dtype: int64
🎯 Distribución en prueba:
Glosa
0    772
1    148
Name: count, dtype: int64


In [ ]:
sm = SMOTE(random_state=45)

X_train, y_train = sm.fit_resample(X_train, y_train)
print("\n--- Después de SMOTE ---")
print(f"Forma de X_train: {X_train.shape}")
print(f"Distribución en y_train:\n{y_train.value_counts()}")


--- Después de SMOTE ---
Forma de X_train: (3600, 40)
Distribución en y_train:
Glosa
1    1800
0    1800
Name: count, dtype: int64


In [ ]:
# RobustScaler
rs = RobustScaler()
rs.fit(X_train)
X_train = rs.transform(X_train)
X_test = rs.transform(X_test)

In [ ]:
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Definir la parrilla de hiperparámetros para XGBoost
param_grid = {
    'n_estimators': [50, 100, 200],      # Número de árboles en el ensamble
    'learning_rate': [0.01, 0.1, 0.2],   # Tasa de aprendizaje
    'max_depth': [3, 4, 5],               # Profundidad máxima de cada árbol
    'eval_metric': ['logloss','aucpr']

}

# 2. Instanciar el modelo XGBClassifier
# Nota: Se añaden use_label_encoder y eval_metric para evitar warnings
model = xgb.XGBClassifier(use_label_encoder=False)

grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_accuracy = grid_search.best_score_

print("Mejor combinación de hiperparámetros:", best_params)
print("Precisión más alta encontrada:", best_accuracy)

Mejor combinación de hiperparámetros: {'eval_metric': 'logloss', 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200}
Precisión más alta encontrada: 0.8483407196706934
